# 04 — Bootstrap CI + cross-domain comparison

Three things:
1. **Sanity-recompute metrics** for every team-split run (E0_team_full, E0_team_clean, E5_team, E7_team, Fusion_team) directly from `team_runs/proba/*.npy` and `team_runs/splits/y_test.npy` (defensive: catches any shape/order drift between Stage 2/Stage 3 and now).
2. **Bootstrap CI** for 7 model pairs on the team test, both R@P90 and PR-AUC (n=1000, seed=42).
3. **Recompute mine metrics** from `fintech_approaches/test_proba_*.npy` + `y_test_canon.npy` for the cross-domain comparison row in `team_split_results.md`. We compute these — not pull from a csv — to avoid any chance of drift between the printed numbers and the actual probas.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ROOT = Path('..').resolve()
REPO = ROOT.parent  # dazzling-vaughan/
sys.path.insert(0, str(ROOT / 'notebooks'))
import utils  # noqa: E402

PROBA_TEAM = ROOT / 'proba'
SPLITS = ROOT / 'splits'
RESULTS = ROOT / 'results'
PROBA_MINE = REPO / 'fintech_approaches'

y_test_team = np.load(SPLITS / 'y_test.npy')
print(f'team y_test: shape={y_test_team.shape}, pos_rate={y_test_team.mean():.4f}')

team y_test: shape=(58410,), pos_rate=0.0795


In [2]:
# ---- Sanity: recompute team metrics from saved probas --------------------
team_probas = {
    'E0_team_clean': np.load(PROBA_TEAM / 'test_proba_e0_team_clean.npy'),
    'E0_team_full':  np.load(PROBA_TEAM / 'test_proba_e0_team_full.npy'),
    'E5_team':       np.load(PROBA_TEAM / 'test_proba_e5_team.npy'),
    'E7_team':       np.load(PROBA_TEAM / 'test_proba_e7_team.npy'),
    'Fusion_team':   np.load(PROBA_TEAM / 'test_proba_fusion_team.npy'),
}
team_metrics = {}
for name, p in team_probas.items():
    assert p.shape == y_test_team.shape, f'{name} shape mismatch: {p.shape} vs y={y_test_team.shape}'
    team_metrics[name] = utils.evaluate(y_test_team, p, name)

E0_team_clean             | ROC-AUC=0.9050 | PR-AUC=0.5301 | R@P90=0.0022
E0_team_full              | ROC-AUC=0.9366 | PR-AUC=0.6812 | R@P90=0.0691
E5_team                   | ROC-AUC=0.9508 | PR-AUC=0.7189 | R@P90=0.0950
E7_team                   | ROC-AUC=0.9521 | PR-AUC=0.7224 | R@P90=0.0915
Fusion_team               | ROC-AUC=0.9522 | PR-AUC=0.7284 | R@P90=0.1077


In [3]:
# ---- Bootstrap CI: 7 pairs x 2 metrics -----------------------------------
PAIRS = [
    ('E0_team_full',  'E0_team_clean'),  # baseline diff: my features vs team features
    ('E5_team',       'E0_team_clean'),  # CLIP gain over team features
    ('E7_team',       'E0_team_clean'),  # text gain over team features
    ('E5_team',       'E0_team_full'),   # CLIP gain over my features
    ('E7_team',       'E0_team_full'),   # text gain over my features
    ('Fusion_team',   'E5_team'),        # is fusion better than CLIP-only?
    ('Fusion_team',   'E7_team'),        # is fusion better than text-only?
]
METRICS = [('r_at_p90', utils.r_at_p90), ('pr_auc', utils.avg_precision_metric)]

boot_records = []
for b, a in PAIRS:  # delta = metric(b) - metric(a)
    for mname, mfn in METRICS:
        res = utils.bootstrap_diff(y_test_team, team_probas[a], team_probas[b],
                                    metric_fn=mfn, n=1000, seed=42)
        boot_records.append({
            'pair': f'{b} vs {a}',
            'metric': mname,
            'delta_mean': res['delta_mean'],
            'ci_low': res['ci_low'],
            'ci_high': res['ci_high'],
            'significant': res['significant'],
            'n_effective': res['n_effective'],
        })
boot_df = pd.DataFrame(boot_records)
boot_df.to_csv(RESULTS / 'bootstrap_team_pairs.csv', index=False)
print(boot_df.to_string(index=False))

                         pair   metric  delta_mean    ci_low  ci_high  significant  n_effective
E0_team_full vs E0_team_clean r_at_p90    0.075604  0.019577 0.143880         True         1000
E0_team_full vs E0_team_clean   pr_auc    0.151012  0.140496 0.161507         True         1000
     E5_team vs E0_team_clean r_at_p90    0.099625  0.064019 0.126989         True         1000
     E5_team vs E0_team_clean   pr_auc    0.188841  0.177395 0.199942         True         1000
     E7_team vs E0_team_clean r_at_p90    0.106425  0.055203 0.147779         True         1000
     E7_team vs E0_team_clean   pr_auc    0.192182  0.180268 0.203799         True         1000
      E5_team vs E0_team_full r_at_p90    0.024021 -0.028350 0.075056        False         1000
      E5_team vs E0_team_full   pr_auc    0.037828  0.033143 0.042667         True         1000
      E7_team vs E0_team_full r_at_p90    0.030821 -0.013001 0.087814        False         1000
      E7_team vs E0_team_full   pr_auc  

In [4]:
# ---- Recompute mine metrics for cross-domain row -------------------------
y_test_mine = np.load(PROBA_MINE / 'y_test_canon.npy')
print(f'mine y_test: shape={y_test_mine.shape}, pos_rate={y_test_mine.mean():.4f}')

mine_probas = {
    'E0_canon (mine)':            np.load(PROBA_MINE / 'test_proba_e0_canon.npy'),
    'E5_CLIP_PCA_25 (mine)':      np.load(PROBA_MINE / 'test_proba_e5.npy'),
    'E7_text_e5small_PCA25 (mine)': np.load(PROBA_MINE / 'test_proba_e7.npy'),
}
mine_metrics = {}
for name, p in mine_probas.items():
    assert p.shape == y_test_mine.shape, f'{name}: {p.shape} vs {y_test_mine.shape}'
    mine_metrics[name] = utils.evaluate(y_test_mine, p, name)

mine y_test: shape=(34416,), pos_rate=0.0593
E0_canon (mine)           | ROC-AUC=0.9208 | PR-AUC=0.6587 | R@P90=0.1721
E5_CLIP_PCA_25 (mine)     | ROC-AUC=0.9232 | PR-AUC=0.6724 | R@P90=0.2956
E7_text_e5small_PCA25 (mine) | ROC-AUC=0.9285 | PR-AUC=0.6660 | R@P90=0.3299


In [5]:
# ---- Persist a single combined report-ready JSON -------------------------
combined = {
    'team_metrics': team_metrics,
    'mine_metrics': mine_metrics,
    'bootstrap_pairs': boot_records,
    'y_test_team_size': int(y_test_team.shape[0]),
    'y_test_team_pos_rate': float(y_test_team.mean()),
    'y_test_mine_size': int(y_test_mine.shape[0]),
    'y_test_mine_pos_rate': float(y_test_mine.mean()),
}
with open(RESULTS / 'team_split_combined.json', 'w') as f:
    json.dump(combined, f, indent=2, default=float)
print(f'saved {RESULTS / "team_split_combined.json"}')

saved /Users/sofya/women-in-data-thesis/.claude/worktrees/dazzling-vaughan/team_runs/results/team_split_combined.json
